## Task 1.1(a): instalar las librerías en el entorno del kernel.

In [1]:
%pip install geopandas pandas numpy matplotlib requests shapely rasterio

  Using cached shapely-2.1.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached pyogrio-0.13.0-cp311-abi3-macosx_12_0_arm64.whl.metadata (5.8 kB)
Using cached shapely-2.1.2-cp312-cp312-macosx_11_0_arm64.whl (1.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.5/23.5 MB 2.8 MB/s  0:00:08m0:00:0100:01
Using cached pyogrio-0.13.0-cp311-abi3-macosx_12_0_arm64.whl (24.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 2.2 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [geopandas]/6 [geopandas]

[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Imports y verificación de versiones

In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import shapely
import rasterio

print("Librerías importadas correctamente.")
print(f"Versión de GeoPandas: {gpd.__version__}")
print(f"Versión de Shapely: {shapely.__version__}")

Librerías importadas correctamente.
Versión de GeoPandas: 1.2.0
Versión de Shapely: 2.1.2


## Task 1.1(b): descargar las divisiones administrativas de Argentina.

In [3]:
from pathlib import Path

# Fuente: GADM 4.1.
# ARG corresponde al código de Argentina.
# Nivel 1: provincias y divisiones equivalentes.
# Nivel 2: departamentos, partidos y divisiones equivalentes.

carpeta_datos = Path("datos")
carpeta_datos.mkdir(parents=True, exist_ok=True)

def descargar_gadm(nivel):
    nombre = f"gadm41_ARG_{nivel}.json"
    url = f"https://geodata.ucdavis.edu/gadm/gadm4.1/json/{nombre}"
    ruta = carpeta_datos / nombre

    print(f"Descargando nivel {nivel} desde: {url}")

    respuesta = requests.get(url, timeout=180)
    respuesta.raise_for_status()
    ruta.write_bytes(respuesta.content)

    capa = gpd.read_file(ruta)

    # Verificar el CRS declarado por la fuente.
    if capa.crs is None:
        raise ValueError(f"El nivel {nivel} no tiene un CRS definido.")

    if capa.crs.to_epsg() != 4326:
        raise ValueError(
            f"Se esperaba EPSG:4326 en el nivel {nivel}, "
            f"pero se encontró {capa.crs}."
        )

    print(f"Nivel {nivel}: {len(capa)} divisiones | CRS: {capa.crs}")
    return capa

gadm_nivel1 = descargar_gadm(1)
gadm_nivel2 = descargar_gadm(2)

Descargando nivel 1 desde: https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_ARG_1.json
Nivel 1: 24 divisiones | CRS: EPSG:4326
Descargando nivel 2 desde: https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_ARG_2.json
Nivel 2: 502 divisiones | CRS: EPSG:4326


## Task 1.1(c): ingresar la llave de Healthsites

In [4]:
from getpass import getpass

healthsites_api_key = getpass("Ingresa tu API key de Healthsites: ").strip()

if not healthsites_api_key:
    raise ValueError("No ingresaste una API key.")

print("Llave almacenada en memoria. Pendiente de validar con la API.")

Llave almacenada en memoria. Pendiente de validar con la API.


In [5]:
url_healthsites = "https://healthsites.io/api/v3/facilities/"

parametros = {
    "api-key": healthsites_api_key,
    "country": "Argentina",
    "page": 1,
    "output": "geojson",
}

try:
    respuesta = requests.get(
        url_healthsites,
        params=parametros,
        timeout=90,
    )
except requests.RequestException:
    raise RuntimeError(
        "No se pudo conectar con Healthsites. Revisa tu conexión e intenta de nuevo."
    ) from None

print(f"Código HTTP: {respuesta.status_code}")

if respuesta.status_code != 200:
    raise RuntimeError(
        f"La API devolvió el código {respuesta.status_code}. "
        "La consulta no se completó."
    )

try:
    datos_prueba = respuesta.json()
except ValueError:
    raise RuntimeError("La respuesta no contiene un JSON válido.") from None

print(f"Tipo de respuesta: {type(datos_prueba).__name__}")

if isinstance(datos_prueba, dict):
    print("Campos principales:", list(datos_prueba.keys()))

    if "features" in datos_prueba:
        print("Instalaciones en esta página:", len(datos_prueba["features"]))

        if datos_prueba["features"]:
            primera = datos_prueba["features"][0]
            print("Campos de una instalación:", list(primera.keys()))
            print(
                "Atributos disponibles:",
                list((primera.get("properties") or {}).keys()),
            )

elif isinstance(datos_prueba, list):
    print("Registros en esta página:", len(datos_prueba))

    if datos_prueba and isinstance(datos_prueba[0], dict):
        print("Campos de una instalación:", list(datos_prueba[0].keys()))

Código HTTP: 403


RuntimeError: La API devolvió el código 403. La consulta no se completó.

## Task 1.1(d): descargar el ráster de población de WorldPop

In [7]:
# Task 1.1(d): descarga programática de WorldPop.
# Producto: Global2 R2025A, versión 1.
# Territorio: Argentina.
# Año: 2030, último disponible en esta serie (proyección futura).
# Resolución nominal: 1 km.
# Unidad de los valores: habitantes por celda.
# Catálogo: https://hub.worldpop.org/geodata/listing?id=136

anio_worldpop = 2030

nombre_worldpop = (
    f"arg_pop_{anio_worldpop}_CN_1km_R2025A_UA_v1.tif"
)

url_worldpop = (
    "https://data.worldpop.org/GIS/Population/"
    f"Global_2015_2030/R2025A/{anio_worldpop}/ARG/"
    f"v1/1km_ua/constrained/{nombre_worldpop}"
)

ruta_worldpop = carpeta_datos / nombre_worldpop
ruta_temporal = ruta_worldpop.with_suffix(".tif.part")

print(f"Descargando población proyectada de Argentina para {anio_worldpop}...")

with requests.get(url_worldpop, stream=True, timeout=180) as descarga:
    descarga.raise_for_status()

    with open(ruta_temporal, "wb") as archivo:
        for bloque in descarga.iter_content(chunk_size=1024 * 1024):
            if bloque:
                archivo.write(bloque)

ruta_temporal.replace(ruta_worldpop)

with rasterio.open(ruta_worldpop) as raster:
    print("GeoTIFF abierto correctamente.")
    print(f"CRS: {raster.crs}")
    print(f"Filas: {raster.height}")
    print(f"Columnas: {raster.width}")

print(f"Archivo guardado en: {ruta_worldpop}")
print(f"Tamaño: {ruta_worldpop.stat().st_size / (1024 ** 2):.2f} MB")

Descargando población proyectada de Argentina para 2030...
GeoTIFF abierto correctamente.
CRS: EPSG:4326
Filas: 3994
Columnas: 2390
Archivo guardado en: datos/arg_pop_2030_CN_1km_R2025A_UA_v1.tif
Tamaño: 5.63 MB


In [8]:
from pyproj import Geod

geod = Geod(ellps="WGS84")

with rasterio.open(ruta_worldpop) as raster:
    poblacion = raster.read(1, masked=True)
    poblacion = np.ma.masked_invalid(poblacion)

    crs_worldpop = raster.crs
    filas, columnas = raster.height, raster.width
    transformacion = raster.transform
    resolucion_x, resolucion_y = raster.res

    if raster.crs.to_epsg() != 4326:
        raise ValueError("Esta celda espera un ráster en EPSG:4326.")

    if transformacion.b != 0 or transformacion.d != 0:
        raise ValueError("Esta celda espera un ráster sin rotación.")

    lon_centro, lat_centro = raster.xy(filas // 2, columnas // 2)

    _, _, ancho_m = geod.inv(
        lon_centro - resolucion_x / 2, lat_centro,
        lon_centro + resolucion_x / 2, lat_centro,
    )
    _, _, alto_m = geod.inv(
        lon_centro, lat_centro - resolucion_y / 2,
        lon_centro, lat_centro + resolucion_y / 2,
    )

    areas_filas_km2 = np.empty(filas)
    oeste = raster.bounds.left
    este = oeste + resolucion_x

    for fila in range(filas):
        norte = transformacion.f + fila * transformacion.e
        sur = norte + transformacion.e

        area_m2, _ = geod.polygon_area_perimeter(
            [oeste, este, este, oeste],
            [norte, norte, sur, sur],
        )
        areas_filas_km2[fila] = abs(area_m2) / 1_000_000

    densidad = poblacion / areas_filas_km2[:, np.newaxis]

    resumen_worldpop = pd.DataFrame([{
        "Año (proyección)": anio_worldpop,
        "CRS": str(crs_worldpop),
        "Filas": filas,
        "Columnas": columnas,
        "Ancho en el centro (m)": round(ancho_m, 2),
        "Alto en el centro (m)": round(alto_m, 2),
        "Máximo de habitantes por celda": round(float(poblacion.max()), 2),
        "Densidad máxima (hab/km²)": round(float(densidad.max()), 2),
    }])

display(resumen_worldpop)

,Año (proyección),CRS,Filas,Columnas,Ancho en el centro (m),Alto en el centro (m),Máximo de habitantes por celda,Densidad máxima (hab/km²)
0,2030,EPSG:4326,3994,2390,727.65,925.04,26718.29,37807.31
